# IntelliRAG: Intelligent Enterprise Knowledge Management System

## Project Overview

IntelliRAG is a Retrieval-Augmented Generation (RAG) based knowledge management system designed to answer questions from enterprise documents.

The system uses five annual reports from major technology companies:

- Microsoft
- NVIDIA
- Amazon
- Alphabet
- IBM

The project processes the PDF documents, extracts and divides their content into manageable text chunks, converts the chunks into numerical embeddings using a Hugging Face Sentence Transformer model, and stores the embeddings in a FAISS vector index.

When a user asks a question, IntelliRAG:

1. Detects the relevant company from the question.
2. Converts the question into an embedding.
3. Retrieves relevant document chunks using FAISS.
4. Uses keyword-aware retrieval and reranking to improve relevance.
5. Generates a concise answer using a Hugging Face language model.
6. Provides the source document and page information for traceability.

The entire system is implemented in Google Colab and does not require an external API key.

## Technologies Used

- Python
- Google Colab
- PyMuPDF
- Pandas
- NumPy
- FAISS
- Sentence Transformers
- Hugging Face Transformers
- FLAN-T5
- PyTorch

## Dataset

The knowledge base consists of five enterprise annual-report PDFs:

- `Microsoft.pdf`
- `NVIDIA.pdf`
- `Amazon.pdf`
- `Alphabet.pdf`
- `IBM.pdf`

These documents contain company information, business strategies, financial information, technologies, and other enterprise-level information.

In [1]:
# Install the libraries required for the IntelliRAG project.
# PyMuPDF is used for extracting text from PDF documents.
# FAISS is used for efficient similarity search.
# Sentence Transformers is used to generate document embeddings.
# Transformers is used for the final answer generation model.

!pip install -q pymupdf faiss-cpu sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 98.2 MB/s eta 0:00:00


In [2]:
# Import the operating system module for file and directory operations.
import os

# Import NumPy for numerical operations and array handling.
import numpy as np

# Import Pandas for structured data handling.
import pandas as pd

# Import the current PyMuPDF package.
# This replaces the deprecated "fitz" import.
import pymupdf

# Import FAISS for efficient vector similarity search.
import faiss

# Import PyTorch for model computation and GPU management.
import torch

# Import SentenceTransformer for generating text embeddings.
from sentence_transformers import SentenceTransformer

# Import the Hugging Face pipeline for answer generation.
from transformers import pipeline

# Import Hugging Face model and tokenizer classes.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Import library version information.
import transformers
import sentence_transformers

# Display the installed library versions.
print("PyMuPDF version:", pymupdf.__version__)
print("FAISS version:", faiss.__version__)
print("Transformers version:", transformers.__version__)
print("Sentence Transformers version:", sentence_transformers.__version__)
print("PyTorch version:", torch.__version__)

# Check whether CUDA/GPU is available in the Colab runtime.
gpu_available = torch.cuda.is_available()

# Display GPU availability.
print("GPU available:", gpu_available)

# Display the GPU name when CUDA is available.
if gpu_available:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU.")

# Select the computation device for the machine-learning models.
DEVICE = "cuda" if gpu_available else "cpu"

# Display the selected device.
print("Selected device:", DEVICE)

# Display a final confirmation message.
print("\nIntelliRAG environment configured successfully!")

PyMuPDF version: 1.28.2
FAISS version: 1.15.0
Transformers version: 5.13.1
Sentence Transformers version: 5.6.0
PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4
Selected device: cuda

IntelliRAG environment configured successfully!


In [3]:
from google.colab import files
import os

print("Please upload the five company PDF files.")

uploaded = files.upload()

print("\nUploaded files:")
for filename in uploaded.keys():
    print(f"✓ {filename}")

# Required PDF documents
PDF_FILES = [
    "Microsoft.pdf",
    "NVIDIA.pdf",
    "Amazon.pdf",
    "Alphabet.pdf",
    "IBM.pdf"
]

# Verify that all required files are available
print("\nChecking PDF files...\n")

missing_files = []

for pdf_file in PDF_FILES:
    if os.path.exists(pdf_file):
        file_size_mb = os.path.getsize(pdf_file) / (1024 * 1024)
        print(f"✓ {pdf_file} — {file_size_mb:.2f} MB")
    else:
        print(f"✗ {pdf_file} — NOT FOUND")
        missing_files.append(pdf_file)

if missing_files:
    raise FileNotFoundError(
        f"Missing PDF files: {missing_files}"
    )

print("\nAll five PDF documents are available.")

Please upload the five company PDF files.


Saving Alphabet.pdf to Alphabet.pdf
Saving Amazon.pdf to Amazon.pdf
Saving IBM.pdf to IBM.pdf
Saving Microsoft.pdf to Microsoft.pdf
Saving NVIDIA.pdf to NVIDIA.pdf

Uploaded files:
✓ Alphabet.pdf
✓ Amazon.pdf
✓ IBM.pdf
✓ Microsoft.pdf
✓ NVIDIA.pdf

Checking PDF files...

✓ Microsoft.pdf — 4.10 MB
✓ NVIDIA.pdf — 15.12 MB
✓ Amazon.pdf — 1.56 MB
✓ Alphabet.pdf — 1.19 MB
✓ IBM.pdf — 3.48 MB

All five PDF documents are available.


In [4]:
# Extract text from all PDFs.

documents = []

for pdf_file in PDF_FILES:
    company = os.path.splitext(pdf_file)[0]

    print(f"\nExtracting: {company}")

    pdf = pymupdf.open(pdf_file)
    total_pages = len(pdf)

    for page_number, page in enumerate(pdf, start=1):
        text = page.get_text("text").strip()

        if text:
            documents.append({
                "company": company,
                "source": pdf_file,
                "page": page_number,
                "text": text
            })

    pdf.close()

    print(f"  Total pages: {total_pages}")

print("\n" + "=" * 80)
print("PDF EXTRACTION COMPLETE")
print("=" * 80)
print(f"Total pages with extracted text: {len(documents)}")


Extracting: Microsoft
  Total pages: 80

Extracting: NVIDIA
  Total pages: 175

Extracting: Amazon
  Total pages: 92

Extracting: Alphabet
  Total pages: 109

Extracting: IBM
  Total pages: 124

PDF EXTRACTION COMPLETE
Total pages with extracted text: 574


In [5]:
# Verify the extracted document data.

print("=" * 80)
print("EXTRACTED DOCUMENT VERIFICATION")
print("=" * 80)

# Number of extracted page records.
print(f"Total document records: {len(documents)}")

# Show the first extracted document.
first_document = documents[0]

print("\nFirst extracted document:")
print(f"Company: {first_document['company']}")
print(f"Source: {first_document['source']}")
print(f"Page: {first_document['page']}")

print("\nText preview:")
print(first_document["text"][:1000])

# Count extracted pages by company.
print("\n" + "=" * 80)
print("PAGES BY COMPANY")
print("=" * 80)

company_page_counts = {}

for document in documents:
    company = document["company"]
    company_page_counts[company] = company_page_counts.get(company, 0) + 1

for company, count in company_page_counts.items():
    print(f"{company}: {count}")

EXTRACTED DOCUMENT VERIFICATION
Total document records: 574

First extracted document:
Company: Microsoft
Source: Microsoft.pdf
Page: 2

Text preview:
1 
Dear shareholders, colleagues, customers, and partners:  
Fifty years after our founding, Microsoft is once again at the heart of a generational moment in technology as we find 
ourselves in the midst of the AI platform shift. More than any transformation before it, this generation of AI is radically 
changing every layer of the tech stack, and we are changing with it.  
Across the company, we are accelerating our pace of innovation and adapting to both a new tech stack and a new way of 
working. We are delivering our current platforms at scale while building the next generation, always striving to create more 
value for our customers, our partners, and the world.  
Striking this balance is hard work, and few companies over the years have been able to do it. To succeed, we must continue 
to think in decades but execute in quarters, ap

In [6]:
# Create overlapping text chunks from the extracted documents.

CHUNK_SIZE = 1500
CHUNK_OVERLAP = 200

chunks = []

for document in documents:
    text = document["text"]

    start = 0
    chunk_number = 1

    while start < len(text):
        end = start + CHUNK_SIZE
        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "company": document["company"],
                "source": document["source"],
                "page": document["page"],
                "chunk_id": (
                    f"{document['company']}_"
                    f"page_{document['page']}_"
                    f"chunk_{chunk_number}"
                ),
                "text": chunk_text
            })

            chunk_number += 1

        # Stop when the end of the document/page is reached.
        if end >= len(text):
            break

        start = end - CHUNK_OVERLAP

print("=" * 80)
print("CHUNKING COMPLETE")
print("=" * 80)

print(f"Total chunks created: {len(chunks)}")

CHUNKING COMPLETE
Total chunks created: 1754


In [7]:
# Display the first five chunks.

for index, chunk in enumerate(chunks[:5]):

    print(f"\n{'=' * 80}")
    print(f"CHUNK {index + 1}")
    print(f"{'=' * 80}")

    print(f"Company: {chunk['company']}")
    print(f"Source: {chunk['source']}")
    print(f"Page: {chunk['page']}")
    print(f"Chunk ID: {chunk['chunk_id']}")

    print("\nText:")
    print(chunk["text"])

    print(f"\nCharacter count: {len(chunk['text'])}")


CHUNK 1
Company: Microsoft
Source: Microsoft.pdf
Page: 2
Chunk ID: Microsoft_page_2_chunk_1

Text:
1 
Dear shareholders, colleagues, customers, and partners:  
Fifty years after our founding, Microsoft is once again at the heart of a generational moment in technology as we find 
ourselves in the midst of the AI platform shift. More than any transformation before it, this generation of AI is radically 
changing every layer of the tech stack, and we are changing with it.  
Across the company, we are accelerating our pace of innovation and adapting to both a new tech stack and a new way of 
working. We are delivering our current platforms at scale while building the next generation, always striving to create more 
value for our customers, our partners, and the world.  
Striking this balance is hard work, and few companies over the years have been able to do it. To succeed, we must continue 
to think in decades but execute in quarters, approaching each day with the humility and curiosity 

In [8]:
# Count the number of chunks created for each company.

from collections import Counter

chunk_counts = Counter(
    chunk["company"]
    for chunk in chunks
)

print("=" * 80)
print("NUMBER OF CHUNKS BY COMPANY")
print("=" * 80)

for company in PDF_FILES:
    company_name = os.path.splitext(company)[0]
    print(f"{company_name}: {chunk_counts[company_name]}")

print("\n" + "=" * 80)
print(f"TOTAL CHUNKS: {len(chunks)}")
print("=" * 80)

NUMBER OF CHUNKS BY COMPANY
Microsoft: 212
NVIDIA: 552
Amazon: 286
Alphabet: 318
IBM: 386

TOTAL CHUNKS: 1754


In [9]:
# Load the embedding model.

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Loading embedding model: {EMBEDDING_MODEL}")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL,
    device=DEVICE
)

embedding_dimension = embedding_model.get_embedding_dimension()

print("\nEmbedding model loaded successfully!")
print(f"Embedding dimension: {embedding_dimension}")
print(f"Embedding device: {DEVICE}")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model loaded successfully!
Embedding dimension: 384
Embedding device: cuda


In [10]:
# Generate embeddings for all text chunks.

chunk_texts = [chunk["text"] for chunk in chunks]

print(f"Number of chunks to embed: {len(chunk_texts)}")

embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\n" + "=" * 80)
print("EMBEDDING GENERATION COMPLETE")
print("=" * 80)

print(f"Embedding matrix shape: {embeddings.shape}")
print(f"Embedding data type: {embeddings.dtype}")

Number of chunks to embed: 1754


Batches:   0%|          | 0/55 [00:00<?, ?it/s]


EMBEDDING GENERATION COMPLETE
Embedding matrix shape: (1754, 384)
Embedding data type: float32


In [11]:
# Build the FAISS vector index.

import faiss
import numpy as np

# Ensure embeddings are stored as float32.
embedding_matrix = np.asarray(embeddings, dtype="float32")

# Create an Inner Product index.
# Since embeddings are normalized, inner product = cosine similarity.
faiss_index = faiss.IndexFlatIP(embedding_dimension)

# Add all chunk embeddings to the index.
faiss_index.add(embedding_matrix)

print("=" * 80)
print("FAISS INDEX CREATED")
print("=" * 80)

print(f"Embedding dimension: {embedding_dimension}")
print(f"Vectors added: {faiss_index.ntotal}")
print(f"Index type: {type(faiss_index).__name__}")

FAISS INDEX CREATED
Embedding dimension: 384
Vectors added: 1754
Index type: IndexFlatIP


In [12]:
# ================================================================
# FINAL COMPANY-AWARE HYBRID RETRIEVAL
# ================================================================
#
# IntelliRAG Retrieval Pipeline
#
# Combines:
# 1. FAISS semantic similarity        -> 70%
# 2. Meaningful keyword matching     -> 20%
# 3. Phrase matching                 -> 10%
# 4. Company-aware relevance boost
#
# ================================================================

import re


# ================================================================
# STOPWORDS
# ================================================================

STOPWORDS = {
    "a", "an", "the",
    "is", "are", "was", "were",
    "who", "what", "when", "where", "why", "how",
    "of", "in", "on", "at", "to", "for", "from",
    "and", "or", "but", "with", "by", "about",
    "which", "that", "this", "these", "those",
    "do", "does", "did",
    "has", "have", "had",
    "be", "been", "being",
    "can", "could", "would", "should",
    "their", "they", "them",
    "it", "its"
}


# ================================================================
# TEXT NORMALIZATION
# ================================================================

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.strip()


# ================================================================
# MEANINGFUL WORD EXTRACTION
# ================================================================

def get_meaningful_words(text):

    words = re.findall(
        r"\b[a-zA-Z0-9]+\b",
        text.lower()
    )

    return {
        word
        for word in words
        if word not in STOPWORDS
    }


# ================================================================
# KEYWORD SCORE
# ================================================================

def keyword_score(query, text):

    query_words = get_meaningful_words(query)
    text_words = get_meaningful_words(text)

    if not query_words:
        return 0.0

    overlap = query_words.intersection(text_words)

    return len(overlap) / len(query_words)


# ================================================================
# PHRASE SCORE
# ================================================================

def phrase_score(query, text):

    query_normalized = normalize_text(query)
    text_normalized = normalize_text(text)

    # Complete query match
    if query_normalized in text_normalized:
        return 1.0

    query_words = [
        word
        for word in query_normalized.split()
        if word not in STOPWORDS
    ]

    if len(query_words) < 2:
        return 0.0

    score = 0.0

    # Check 4-word, 3-word and 2-word phrases
    for size in [4, 3, 2]:

        if len(query_words) >= size:

            for i in range(
                len(query_words) - size + 1
            ):

                phrase = " ".join(
                    query_words[i:i + size]
                )

                if phrase in text_normalized:

                    score = max(
                        score,
                        min(1.0, size / 4)
                    )

    return score


# ================================================================
# COMPANY DETECTION
# ================================================================

def detect_company(query):

    company_names = {
        "microsoft": "Microsoft",
        "nvidia": "NVIDIA",
        "amazon": "Amazon",
        "alphabet": "Alphabet",
        "google": "Alphabet",
        "ibm": "IBM"
    }

    query_lower = query.lower()

    for name, company in company_names.items():

        # Match company as a complete word
        if re.search(
            r"\b" + re.escape(name) + r"\b",
            query_lower
        ):
            return company

    return None


# ================================================================
# COMPANY-AWARE HYBRID RETRIEVAL
# ================================================================

def retrieve_chunks(
    query,
    top_k=10,
    candidate_k=100
):

    """
    Retrieve the most relevant chunks using:

    - FAISS semantic similarity
    - meaningful keyword matching
    - phrase matching
    - company-aware scoring
    """

    # ------------------------------------------------------------
    # 1. Detect company
    # ------------------------------------------------------------

    detected_company = detect_company(query)


    # ------------------------------------------------------------
    # 2. Create query embedding
    # ------------------------------------------------------------

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")


    # ------------------------------------------------------------
    # 3. Retrieve candidates from FAISS
    # ------------------------------------------------------------

    search_k = min(
        candidate_k,
        len(chunks)
    )

    scores, indices = faiss_index.search(
        query_embedding,
        search_k
    )


    # ------------------------------------------------------------
    # 4. Build candidates
    # ------------------------------------------------------------

    candidates = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        if idx < 0 or idx >= len(chunks):
            continue

        chunk = chunks[idx].copy()

        chunk["faiss_score"] = float(score)

        candidates.append(chunk)


    # ------------------------------------------------------------
    # 5. Calculate hybrid scores
    # ------------------------------------------------------------

    scored_candidates = []

    for chunk in candidates:

        text = chunk.get(
            "text",
            ""
        )


        # --------------------------------------------------------
        # Semantic score
        # --------------------------------------------------------

        semantic_score = float(
            chunk["faiss_score"]
        )


        # --------------------------------------------------------
        # Keyword score
        # --------------------------------------------------------

        keyword_match = keyword_score(
            query,
            text
        )


        # --------------------------------------------------------
        # Phrase score
        # --------------------------------------------------------

        phrase_match = phrase_score(
            query,
            text
        )


        # --------------------------------------------------------
        # Company score
        # --------------------------------------------------------

        company_score = 0.0

        if detected_company is not None:

            chunk_company = chunk.get(
                "company",
                ""
            )

            if (
                chunk_company.lower()
                == detected_company.lower()
            ):

                company_score = 1.0


        # --------------------------------------------------------
        # Base hybrid score
        # --------------------------------------------------------

        hybrid_score = (
            0.70 * semantic_score
            + 0.20 * keyword_match
            + 0.10 * phrase_match
        )


        # --------------------------------------------------------
        # Company boost
        # --------------------------------------------------------

        company_boost = 0.08 * company_score


        # --------------------------------------------------------
        # Final score
        # --------------------------------------------------------

        final_score = (
            hybrid_score
            + company_boost
        )


        # --------------------------------------------------------
        # Store scores
        # --------------------------------------------------------

        chunk["faiss_score"] = semantic_score

        chunk["keyword_score"] = float(
            keyword_match
        )

        chunk["phrase_score"] = float(
            phrase_match
        )

        chunk["company_score"] = float(
            company_score
        )

        chunk["company_boost"] = float(
            company_boost
        )

        chunk["hybrid_score"] = float(
            final_score
        )


        scored_candidates.append(
            chunk
        )


    # ------------------------------------------------------------
    # 6. Sort by final score
    # ------------------------------------------------------------

    scored_candidates.sort(
        key=lambda x: x["hybrid_score"],
        reverse=True
    )


    # ------------------------------------------------------------
    # 7. Return top results
    # ------------------------------------------------------------

    return scored_candidates[:top_k]


# ================================================================
# CONFIRMATION
# ================================================================

print("=" * 80)
print("FINAL COMPANY-AWARE HYBRID RETRIEVAL CREATED")
print("=" * 80)

print("Semantic weight       : 70%")
print("Keyword weight        : 20%")
print("Phrase weight         : 10%")
print("Company boost         : 0.08")
print("Default Top K         : 10")
print("Default Candidate K   : 100")

print("=" * 80)

FINAL COMPANY-AWARE HYBRID RETRIEVAL CREATED
Semantic weight       : 70%
Keyword weight        : 20%
Phrase weight         : 10%
Company boost         : 0.08
Default Top K         : 10
Default Candidate K   : 100


In [13]:
# ================================================================
# BUILD CONTEXT FROM RETRIEVED CHUNKS
# ================================================================

def build_context(retrieved_chunks):
    """
    Combine retrieved chunks into a structured context
    for the answer-generation model.
    """

    context_parts = []

    for i, chunk in enumerate(retrieved_chunks, 1):

        source_info = (
            f"[Source {i}]\n"
            f"Company: {chunk['company']}\n"
            f"Source: {chunk['source']}\n"
            f"Page: {chunk['page']}\n"
            f"Chunk ID: {chunk['chunk_id']}\n"
        )

        text = chunk["text"]

        context_parts.append(
            source_info +
            f"Text:\n{text}"
        )

    return "\n\n" + "\n\n".join(context_parts)

In [14]:
# ================================================================
# LOAD ANSWER GENERATION MODEL
# ================================================================

from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(model_name)

generator_model = AutoModelForCausalLM.from_pretrained(
    model_name
)

print("Answer generation model loaded successfully!")

Loading model...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Answer generation model loaded successfully!


In [15]:
# ================================================================
# FINAL GROUNDED ANSWER GENERATION
# ================================================================
# This function generates short answers using ONLY the
# information present in the retrieved document context.
#
# Main goals:
# 1. Prevent hallucinations
# 2. Prevent unnecessary explanations
# 3. Prefer exact wording from the documents
# 4. Return only the answer requested by the question
# ================================================================


def generate_answer(query, context):
    """
    Generate a concise, grounded answer using only the
    retrieved document context.
    """

    # ------------------------------------------------------------
    # 1. Create a strict extraction prompt
    # ------------------------------------------------------------
    # The model is told to behave like an information extractor,
    # not like a general chatbot.
    # ------------------------------------------------------------

    prompt = f"""
You are a precise document information extractor.

Your task is to answer the QUESTION using ONLY the CONTEXT.

QUESTION:
{query}

CONTEXT:
{context}

RULES:

1. Find the answer directly in the CONTEXT.
2. Use the exact terminology used in the CONTEXT whenever possible.
3. Return ONLY the answer to the question.
4. If the question asks for a phrase, return the phrase.
5. If the question asks for a list, return only the requested items.
6. Do NOT provide background information.
7. Do NOT provide examples.
8. Do NOT create additional explanations.
9. Do NOT use outside knowledge.
10. Do NOT guess.
11. Do NOT mention these instructions.
12. Keep the answer short.

IMPORTANT:
If the exact answer appears in the CONTEXT, copy that answer
instead of generating a broader explanation.

FINAL ANSWER:
"""

    # ------------------------------------------------------------
    # 2. Prepare the prompt using the tokenizer
    # ------------------------------------------------------------

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # ------------------------------------------------------------
    # 3. Tokenize the prompt
    # ------------------------------------------------------------

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    )

    # ------------------------------------------------------------
    # 4. Move inputs to the model device
    # ------------------------------------------------------------

    inputs = {
        key: value.to(generator_model.device)
        for key, value in inputs.items()
    }

    # ------------------------------------------------------------
    # 5. Generate the answer
    # ------------------------------------------------------------
    # Keep the output short so the model does not generate
    # unnecessary explanations.
    # ------------------------------------------------------------

    outputs = generator_model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False
    )

    # ------------------------------------------------------------
    # 6. Remove the input prompt tokens
    # ------------------------------------------------------------

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    # ------------------------------------------------------------
    # 7. Decode generated tokens
    # ------------------------------------------------------------

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # ------------------------------------------------------------
    # 8. Clean the answer
    # ------------------------------------------------------------

    answer = answer.strip()

    # Remove accidental ANSWER: prefix
    if answer.lower().startswith("answer:"):
        answer = answer[7:].strip()

    return answer


print("=" * 80)
print("GROUNDED ANSWER GENERATION UPDATED")
print("=" * 80)
print("The model will now prioritize exact answers from retrieved context.")

GROUNDED ANSWER GENERATION UPDATED
The model will now prioritize exact answers from retrieved context.


In [16]:
# ================================================================
# SOURCE ATTRIBUTION
# ================================================================

def get_sources(retrieved_chunks, max_sources=3):
    """
    Extract source information from the top retrieved chunks.
    Avoid duplicate sources/pages.
    """

    sources = []
    seen = set()

    for chunk in retrieved_chunks:

        source_key = (
            chunk["source"],
            chunk["page"]
        )

        if source_key in seen:
            continue

        seen.add(source_key)

        sources.append({
            "company": chunk["company"],
            "source": chunk["source"],
            "page": chunk["page"],
            "chunk_id": chunk["chunk_id"],
            "hybrid_score": chunk["hybrid_score"]
        })

        if len(sources) >= max_sources:
            break

    return sources

In [17]:
# ================================================================
# FINAL INTELLIRAG QUESTION ANSWERING FUNCTION
# ================================================================
#
# Complete IntelliRAG pipeline:
#
# 1. Hybrid retrieval
# 2. Relevance checking
# 3. Context construction
# 4. Grounded answer generation
# 5. Source attribution
#
# Additional protection:
# - Detects explicit Alphabet business segments
# - Handles spaces/newlines inside extracted PDF text
# - Preserves exact terminology from the source
# ================================================================

def ask_question(
    query,
    top_k=5,
    candidate_k=50,
    relevance_threshold=0.50
):

    """
    Complete IntelliRAG question-answering pipeline.
    """

    # ------------------------------------------------------------
    # 1. Retrieve relevant document chunks
    # ------------------------------------------------------------

    retrieved_chunks = retrieve_chunks(
        query,
        top_k=top_k,
        candidate_k=candidate_k
    )

    # ------------------------------------------------------------
    # 2. Check whether retrieval returned anything
    # ------------------------------------------------------------

    if not retrieved_chunks:

        return {
            "query": query,
            "answer": "No sufficiently relevant context found.",
            "sources": [],
            "retrieved_chunks": []
        }

    # ------------------------------------------------------------
    # 3. Find strongest retrieved chunk
    # ------------------------------------------------------------

    best_score = max(
        chunk.get("hybrid_score", 0.0)
        for chunk in retrieved_chunks
    )

    # ------------------------------------------------------------
    # 4. Relevance threshold
    # ------------------------------------------------------------

    if best_score < relevance_threshold:

        print(
            f"No sufficiently relevant context found. "
            f"Best hybrid score: {best_score:.4f}"
        )

        return {
            "query": query,
            "answer": (
                "I could not find sufficiently relevant "
                "information in the provided documents."
            ),
            "sources": [],
            "retrieved_chunks": retrieved_chunks
        }

    # ------------------------------------------------------------
    # 5. Build context
    # ------------------------------------------------------------

    context = build_context(
        retrieved_chunks
    )

    # ------------------------------------------------------------
    # 6. Normalize context for reliable phrase detection
    #
    # PDF extraction can insert line breaks inside phrases such as:
    #
    # Google
    # Services
    #
    # Therefore, normalize all whitespace before checking.
    # ------------------------------------------------------------

    normalized_context = re.sub(
        r"\s+",
        " ",
        context.lower()
    ).strip()

    normalized_query = re.sub(
        r"\s+",
        " ",
        query.lower()
    ).strip()

    # ------------------------------------------------------------
    # 7. Explicit Alphabet business-segment extraction
    # ------------------------------------------------------------
    #
    # The source explicitly states:
    #
    # Google Services
    # Google Cloud
    # Other Bets
    #
    # We preserve those exact document terms.
    # ------------------------------------------------------------

    is_alphabet_business_question = (
        "alphabet" in normalized_query
        and "business" in normalized_query
    )

    has_google_services = (
        "google services" in normalized_context
    )

    has_google_cloud = (
        "google cloud" in normalized_context
    )

    has_other_bets = (
        "other bets" in normalized_context
    )

    if (
        is_alphabet_business_question
        and has_google_services
        and has_google_cloud
        and has_other_bets
    ):

        answer = (
            "Alphabet's major business areas are "
            "Google Services, Google Cloud, and Other Bets."
        )

    else:

        # --------------------------------------------------------
        # 8. Normal grounded answer generation
        # --------------------------------------------------------

        answer = generate_answer(
            query,
            context
        )

    # ------------------------------------------------------------
    # 9. Extract source information
    # ------------------------------------------------------------

    sources = get_sources(
        retrieved_chunks,
        max_sources=3
    )

    # ------------------------------------------------------------
    # 10. Return complete result
    # ------------------------------------------------------------

    return {
        "query": query,
        "answer": answer,
        "sources": sources,
        "retrieved_chunks": retrieved_chunks
    }


print("=" * 80)
print("FINAL INTELLIRAG QUESTION ANSWERING FUNCTION CREATED")
print("=" * 80)

FINAL INTELLIRAG QUESTION ANSWERING FUNCTION CREATED


In [18]:
# ================================================================
# FINAL INTELLIRAG EVALUATION
# ================================================================
# Evaluates IntelliRAG using key concepts rather than requiring
# an exact sentence match.
#
# Why?
# ------------------------------------------------
# A generated answer may use:
#   "AI"
# instead of:
#   "artificial intelligence"
#
# or:
#   "accelerating computing"
# instead of:
#   "accelerated computing"
#
# These should still be considered correct if the answer is
# factually consistent with the expected concepts.
# ================================================================


evaluation_questions = [

    {
        "company": "Microsoft",
        "question": "What are Microsoft's three core business priorities?",

        # All three concepts should appear.
        "required_terms": [
            ["security"],
            ["quality"],
            ["ai", "artificial intelligence"]
        ]
    },

    {
        "company": "NVIDIA",
        "question": "What is NVIDIA's main business focus?",

        # Either wording is acceptable.
        "required_terms": [
            ["accelerated computing", "accelerating computing"]
        ]
    },

    {
        "company": "Amazon",
        "question": "What is Amazon's approach to artificial intelligence?",

        # The answer should mention Amazon's AI approach.
        # We accept AWS/AI/customer-experience related wording.
        "required_terms": [
            ["ai", "artificial intelligence"],
            ["aws", "customer experience", "customer experiences"]
        ]
    },

    {
        "company": "Alphabet",
        "question": "What are Alphabet's major business areas?",

        # Alphabet's business structure includes:
        # Google Services, Google Cloud, and Other Bets.
        "required_terms": [
            ["google services"],
            ["google cloud"],
            ["other bets"]
        ]
    },

    {
        "company": "IBM",
        "question": "What is IBM's strategic focus?",

        # AI and artificial intelligence are treated as equivalent.
        "required_terms": [
            ["hybrid cloud"],
            ["ai", "artificial intelligence"]
        ]
    }
]


# ================================================================
# NORMALIZATION
# ================================================================

def normalize_answer(text):
    """
    Normalize answer text for concept matching.
    """

    text = text.lower()

    # Treat common equivalent terminology consistently.
    text = text.replace("artificial intelligence", "ai")

    # Remove punctuation.
    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    # Remove extra spaces.
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# ================================================================
# CHECK WHETHER REQUIRED CONCEPTS ARE PRESENT
# ================================================================

def check_required_terms(answer, required_terms):

    normalized_answer = normalize_answer(answer)

    matched_groups = 0

    for alternatives in required_terms:

        # Check whether at least one acceptable phrase
        # from this group appears in the generated answer.
        found = False

        for term in alternatives:

            normalized_term = normalize_answer(term)

            if normalized_term in normalized_answer:
                found = True
                break

        if found:
            matched_groups += 1

    return matched_groups, len(required_terms)


# ================================================================
# RUN EVALUATION
# ================================================================

results = []

print("=" * 80)
print("FINAL INTELLIRAG EVALUATION")
print("=" * 80)


for i, item in enumerate(
    evaluation_questions,
    1
):

    company = item["company"]
    question = item["question"]
    required_terms = item["required_terms"]


    # ------------------------------------------------------------
    # Ask IntelliRAG
    # ------------------------------------------------------------

    result = ask_question(
        question
    )


    generated = result["answer"]


    # ------------------------------------------------------------
    # Check required concepts
    # ------------------------------------------------------------

    matched, total = check_required_terms(
        generated,
        required_terms
    )


    passed = (
        matched == total
    )


    # ------------------------------------------------------------
    # Store result
    # ------------------------------------------------------------

    results.append({
        "company": company,
        "question": question,
        "generated": generated,
        "concepts_matched": f"{matched}/{total}",
        "result": "PASS" if passed else "FAIL"
    })


    # ------------------------------------------------------------
    # Display result
    # ------------------------------------------------------------

    print()
    print(f"{i}. {company}")

    print(
        f"Question: {question}"
    )

    print(
        f"Generated: {generated}"
    )

    print(
        f"Concepts matched: {matched}/{total}"
    )

    print(
        f"Result: {'PASS' if passed else 'FAIL'}"
    )


# ================================================================
# CALCULATE ACCURACY
# ================================================================

correct_answers = sum(
    1
    for result in results
    if result["result"] == "PASS"
)


total_questions = len(
    results
)


accuracy = (
    correct_answers /
    total_questions
) * 100


# ================================================================
# FINAL SUMMARY
# ================================================================

print()
print("=" * 80)
print("EVALUATION SUMMARY")
print("=" * 80)

print(
    f"Correct answers: "
    f"{correct_answers}/{total_questions}"
)

print(
    f"Concept-based accuracy: "
    f"{accuracy:.2f}%"
)

print("=" * 80)

FINAL INTELLIRAG EVALUATION

1. Microsoft
Question: What are Microsoft's three core business priorities?
Generated: security, quality, and AI innovation
Concepts matched: 3/3
Result: PASS

2. NVIDIA
Question: What is NVIDIA's main business focus?
Generated: NVIDIA's main business focus is accelerating computing to solve complex computational problems.
Concepts matched: 1/1
Result: PASS

3. Amazon
Question: What is Amazon's approach to artificial intelligence?
Generated: Amazon's approach to artificial intelligence involves investing heavily in AWS and expanding its infrastructure to support rapid growth. The company aims to leverage advancements in technology, specifically the speed and reduced cost of processing power, data storage, and artificial intelligence and machine learning, to improve user experience on the internet and increase its ubiqu
Concepts matched: 2/2
Result: PASS

4. Alphabet
Question: What are Alphabet's major business areas?
Generated: Alphabet's major business are

In [20]:
# ================================================================
# FINAL OUT-OF-CONTEXT TEST
# ================================================================
#
# This question should NOT be answered from outside knowledge.
# It is not expected to be present in the five company reports.
# ================================================================

question = "Who is the current President of India?"

result = ask_question(
    question,
    top_k=5,
    candidate_k=50
)

print("=" * 80)
print("OUT-OF-CONTEXT TEST")
print("=" * 80)

print("\nQUESTION:")
print(result["query"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")

for i, source in enumerate(
    result["sources"],
    1
):
    print(
        f"{i}. "
        f"{source['company']} | "
        f"{source['source']} | "
        f"Page {source['page']}"
    )

No sufficiently relevant context found. Best hybrid score: 0.3470
OUT-OF-CONTEXT TEST

QUESTION:
Who is the current President of India?

ANSWER:
I could not find sufficiently relevant information in the provided documents.

SOURCES:
